In [12]:
import torch, platform, subprocess, sys
print("Python:", platform.python_version())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("⚠️ No GPU. In Colab, go to Runtime → Change runtime type → GPU.")


Python: 3.12.11
CUDA available: True
GPU: Tesla T4
Fri Oct 10 04:25:24 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   71C    P0             32W /   70W |    3036MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A

In [13]:
!pip -q install --upgrade pip
!pip -q install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip -q install triton==3.* tensorboard datasets einops
import torch, triton
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda, "| Triton:", triton.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch: 2.8.0+cu126 | CUDA: 12.6 | Triton: 3.4.0
CUDA available: True


In [14]:
import torch, math
torch.manual_seed(0)

# Toy language modeling: random token ids (vocab 4096)
V = 4096
SEQ = 1024
N_BATCHES = 200  # small for demo

def toy_batch(bs=8, seq=SEQ, device="cpu"):
    x = torch.randint(0, V, (bs, seq), device=device)
    y = torch.randint(0, V, (bs, seq), device=device)
    return x, y

# Create pinned CPU buffers to simulate a DataLoader with pin_memory=True
def make_pinned_batch(bs=8, seq=SEQ):
    x, y = toy_batch(bs, seq, device="cpu")
    return x.pin_memory(), y.pin_memory()


In [15]:
import torch
import torch.nn as nn
from einops import rearrange

class TinyBlock(nn.Module):
    def __init__(self, d_model=512, n_heads=8, vocab=4096):
        super().__init__()
        self.tok = nn.Embedding(vocab, d_model)
        self.qkv = nn.Linear(d_model, 3*d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4*d_model),
            nn.GELU(),
            nn.Linear(4*d_model, d_model)
        )
        self.head = nn.Linear(d_model, vocab, bias=False)
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

    def forward(self, x):
        h = self.tok(x)  # [B,S,D]
        B,S,D = h.shape
        qkv = self.qkv(h).view(B,S,3,self.n_heads,self.d_head)
        q,k,v = qkv.unbind(dim=2)  # [B,S,H,dh]
        # scaled dot-product attention (no masking for demo)
        att = torch.softmax((q @ k.transpose(-1,-2)) / math.sqrt(self.d_head), dim=-1)  # [B,S,H,S]
        h = att @ v                                         # [B,S,H,dh]
        h = rearrange(h, 'b s h d -> b s (h d)')
        h = self.proj(h)
        h = h + self.ffn(h)
        logits = self.head(h)   # [B,S,V]
        return logits

def xent(logits, y):
    # flatten to [B*S, V]
    return torch.nn.functional.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))


In [16]:
import time
device = "cuda" if torch.cuda.is_available() else "cpu"
model = TinyBlock().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)

BS = 8
tokens = 0
t0 = time.time()

for step in range(20):
    # CPU batch (no pinning/non_blocking)
    x_cpu, y_cpu = toy_batch(BS, device="cpu")
    x = x_cpu.to(device)  # blocking copy
    y = y_cpu.to(device)

    opt.zero_grad(set_to_none=True)
    logits = model(x)
    loss = xent(logits, y)
    loss.backward()
    opt.step()

    tokens += BS * x.shape[1]
    if (step+1) % 5 == 0:
        dt = time.time() - t0
        print(f"[baseline] step {step+1:03d}  loss={loss.item():.3f}  toks/sec={tokens/dt:.0f}")


[baseline] step 005  loss=8.321  toks/sec=115226
[baseline] step 010  loss=8.320  toks/sec=111986
[baseline] step 015  loss=8.320  toks/sec=111559
[baseline] step 020  loss=8.321  toks/sec=109411


In [17]:
import time, torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = TinyBlock().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)

BS = 8
tokens = 0
t0 = time.time()

for step in range(20):
    # Pinned host buffers (simulates DataLoader(pin_memory=True))
    x_cpu, y_cpu = make_pinned_batch(BS)
    # Non-blocking H2D (ready for overlap if/when we add streams)
    x = x_cpu.to(device, non_blocking=True)
    y = y_cpu.to(device, non_blocking=True)

    opt.zero_grad(set_to_none=True)
    logits = model(x)
    loss = xent(logits, y)
    loss.backward()
    opt.step()

    tokens += BS * x.shape[1]
    if (step+1) % 5 == 0:
        dt = time.time() - t0
        print(f"[pinned+nb] step {step+1:03d}  loss={loss.item():.3f}  toks/sec={tokens/dt:.0f}")


[pinned+nb] step 005  loss=8.320  toks/sec=2003530
[pinned+nb] step 010  loss=8.319  toks/sec=208129
[pinned+nb] step 015  loss=8.321  toks/sec=161683
[pinned+nb] step 020  loss=8.320  toks/sec=140318


In [18]:
import torch, time, math

# --- tiny model & loss (same as before) ---
class TinyBlock(torch.nn.Module):
    def __init__(self, d_model=512, n_heads=8, vocab=4096):
        super().__init__()
        self.tok = torch.nn.Embedding(vocab, d_model)
        self.qkv = torch.nn.Linear(d_model, 3*d_model, bias=False)
        self.proj = torch.nn.Linear(d_model, d_model, bias=False)
        self.ffn = torch.nn.Sequential(
            torch.nn.Linear(d_model, 4*d_model),
            torch.nn.GELU(),
            torch.nn.Linear(4*d_model, d_model)
        )
        self.head = torch.nn.Linear(d_model, vocab, bias=False)
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

    def forward(self, x):
        h = self.tok(x)  # [B,S,D]
        B,S,D = h.shape
        qkv = self.qkv(h).view(B,S,3,self.n_heads,self.d_head)
        q,k,v = qkv.unbind(dim=2)
        att = torch.softmax((q @ k.transpose(-1,-2)) / math.sqrt(self.d_head), dim=-1)
        h = att @ v
        h = h.reshape(B,S,-1)
        h = self.proj(h)
        h = h + self.ffn(h)
        return self.head(h)

def xent(logits, y):
    return torch.nn.functional.cross_entropy(
        logits.view(-1, logits.size(-1)), y.view(-1)
    )

# --- data helpers (pinned batches) ---
V, SEQ = 4096, 1024
def make_pinned_batch(bs=8, seq=SEQ):
    x = torch.randint(0, V, (bs, seq), device="cpu").pin_memory()
    y = torch.randint(0, V, (bs, seq), device="cpu").pin_memory()
    return x, y

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "Enable GPU runtime in Colab (Runtime → Change runtime type → GPU)."

BS = 8
model = TinyBlock().to(device)
# IMPORTANT: capturable=True for CUDA Graphs
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, capturable=True)

# Pre-allocate STATIC buffers used in capture (no new allocations inside graph)
x_static = torch.empty(BS, SEQ, dtype=torch.long, device=device)
y_static = torch.empty(BS, SEQ, dtype=torch.long, device=device)
loss_buf = torch.zeros((), device=device)  # scalar loss “slot” to write into

torch.cuda.synchronize()

# Warmup outside capture to build caches & allocs
for _ in range(3):
    xb, yb = make_pinned_batch(BS)
    x_static.copy_(xb, non_blocking=True)
    y_static.copy_(yb, non_blocking=True)
    opt.zero_grad(set_to_none=True)
    l = xent(model(x_static), y_static)
    l.backward()
    opt.step()

torch.cuda.synchronize()

# CAPTURE
g = torch.cuda.CUDAGraph()
opt.zero_grad(set_to_none=True)   # grads must be allocated before capture
with torch.cuda.graph(g):
    logits = model(x_static)
    l = xent(logits, y_static)
    loss_buf.copy_(l)             # write into pre-made scalar buffer
    l.backward()
    opt.step()

# REPLAY with new data by copying into x_static/y_static
tokens = 0
t0 = time.time()
for step in range(50):
    xb, yb = make_pinned_batch(BS)
    # These copies happen BEFORE replay on the default stream
    x_static.copy_(xb, non_blocking=True)
    y_static.copy_(yb, non_blocking=True)
    # Replay the captured graph (single launch)
    g.replay()
    tokens += BS * SEQ
    if (step+1) % 10 == 0:
        dt = time.time() - t0
        print(f"[graphs] step {step+1:03d}  loss≈{loss_buf.item():.3f}  toks/sec={tokens/dt:.0f}")


[graphs] step 010  loss≈8.320  toks/sec=13715915
[graphs] step 020  loss≈8.322  toks/sec=221284
[graphs] step 030  loss≈8.322  toks/sec=161979
[graphs] step 040  loss≈8.321  toks/sec=142427
[graphs] step 050  loss≈8.319  toks/sec=133018


In [19]:
# kernels/rmsnorm_triton.py (fixed)
import triton
import triton.language as tl
import torch

@triton.jit
def _rmsnorm_fwd(x_ptr, w_ptr, y_ptr, D, eps,  # pointers + sizes
                 stride_x_bs, stride_y_bs,     # strides across batch*seq rows
                 BLOCK_SIZE: tl.constexpr):
    """
    Each program instance handles one row of length D.
    We iterate over the row in chunks of BLOCK_SIZE.
    """
    pid = tl.program_id(0)  # which row (B*S) we are on
    row_x = x_ptr + pid * stride_x_bs
    row_y = y_ptr + pid * stride_y_bs

    # First pass: compute sum of squares
    acc = tl.zeros((), dtype=tl.float32)
    for offs in range(0, D, BLOCK_SIZE):
        idx = offs + tl.arange(0, BLOCK_SIZE)
        mask = idx < D
        x = tl.load(row_x + idx, mask=mask, other=0.0).to(tl.float32)
        acc += tl.sum(x * x, axis=0)

    mean_sq = acc / D
    inv_rms = tl.rsqrt(mean_sq + eps)

    # Second pass: write normalized * weight
    for offs in range(0, D, BLOCK_SIZE):
        idx = offs + tl.arange(0, BLOCK_SIZE)
        mask = idx < D
        x = tl.load(row_x + idx, mask=mask, other=0.0).to(tl.float32)
        w = tl.load(w_ptr + idx, mask=mask, other=1.0).to(tl.float32)
        y = (x * inv_rms) * w
        tl.store(row_y + idx, y, mask=mask)


def rmsnorm_triton(x: torch.Tensor, w: torch.Tensor, eps: float = 1e-5):
    """
    x: [B, S, D] (float32/bf16/float16), w: [D]
    returns y with same dtype as x
    """
    assert x.is_cuda and w.is_cuda, "Use CUDA tensors"
    B, S, D = x.shape
    y = torch.empty_like(x)

    # Flatten batch and sequence into a single rows dimension
    x_2d = x.view(-1, D)
    y_2d = y.view(-1, D)
    rows = x_2d.shape[0]

    # Choose a reasonable block size (power of two up to 1024)
    BLOCK_SIZE = min(1024, 1 << (D - 1).bit_length())
    grid = (rows,)

    _rmsnorm_fwd[grid](
        x_2d, w, y_2d, D, eps,
        x_2d.stride(0), y_2d.stride(0),
        BLOCK_SIZE=BLOCK_SIZE,
        num_warps=4,
        num_stages=2,
    )
    return y


In [20]:
# device must be set, e.g.:
device = "cuda" if torch.cuda.is_available() else "cpu"

B, S, D = 2, 4, 512
x = torch.randn(B, S, D, device=device, dtype=torch.float32)
w = torch.ones(D, device=device, dtype=torch.float32)

y_ref = torch.nn.functional.layer_norm(x, (D,), weight=w, bias=None, eps=1e-5)
y_tr  = rmsnorm_triton(x, w, eps=1e-5)

print("RMSNorm max |Δ|:", (y_ref - y_tr).abs().max().item())


RMSNorm max |Δ|: 0.06912636756896973
